# 現代卷積神經網路架構 (Modern CNN Architectures)
:label:`sec_modern_architectures`

從2016年ResNet之後，CNN架構設計經歷了重大變革。本節介紹2016-2024年間的重要CNN架構創新。

## 為什麼需要現代架構？

傳統架構（AlexNet, VGG, ResNet）存在的問題：
1. **計算效率低**：參數量大，推理慢
2. **不適合移動端**：需要大量GPU資源
3. **設計靠經驗**：缺乏系統化的設計方法
4. **缺乏可解釋性**：不清楚為什麼某些設計有效

現代架構的創新方向：
- ⚡ **效率優化**：MobileNet, ShuffleNet
- 🔍 **注意力機制**：SENet, CBAM
- 🎯 **神經架構搜索**：EfficientNet, NAS
- 🏗️ **架構創新**：ResNeXt, Xception

## 本節內容

1. **SENet**：通道注意力機制
2. **MobileNet 系列**：輕量級網路設計
3. **EfficientNet**：複合縮放與NAS
4. **ResNeXt**：分組卷積與基數
5. **性能對比與應用場景**

## 1. 環境準備

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torch.utils.data import DataLoader

import torchvision
from torchvision import transforms, models
import timm

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import time

# 設定設備
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'使用設備: {device}')

# 設定樣式
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

## 2. SENet (Squeeze-and-Excitation Networks)

**論文**: [Squeeze-and-Excitation Networks](https://arxiv.org/abs/1709.01507) (CVPR 2018, ImageNet 2017 冠軍)

### 2.1 核心思想

SENet 引入了**通道注意力機制**（Channel Attention），讓網路學習不同特徵通道的重要性。

#### SE 模塊工作流程：

```
輸入特徵圖 [B, C, H, W]
    ↓
1. Squeeze: 全局平均池化 → [B, C, 1, 1]
    ↓
2. Excitation: FC → ReLU → FC → Sigmoid → [B, C, 1, 1]
    ↓
3. Scale: 通道加權 → [B, C, H, W]
```

### 2.2 實現 SE 模塊

In [ ]:
class SEBlock(nn.Module):
    """Squeeze-and-Excitation Block"""
    
    def __init__(self, channels, reduction=16):
        """
        Args:
            channels: 輸入通道數
            reduction: 降維比例（減少計算量）
        """
        super(SEBlock, self).__init__()
        
        # Squeeze: 全局平均池化
        self.squeeze = nn.AdaptiveAvgPool2d(1)
        
        # Excitation: 兩層全連接
        self.excitation = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        b, c, _, _ = x.size()
        
        # Squeeze: [B, C, H, W] → [B, C, 1, 1] → [B, C]
        y = self.squeeze(x).view(b, c)
        
        # Excitation: [B, C] → [B, C]
        y = self.excitation(y).view(b, c, 1, 1)
        
        # Scale: 通道加權
        return x * y.expand_as(x)

# 測試 SE Block
se_block = SEBlock(channels=64, reduction=16)
x = torch.randn(2, 64, 32, 32)
output = se_block(x)
print(f"輸入形狀: {x.shape}")
print(f"輸出形狀: {output.shape}")
print(f"SE Block 參數量: {sum(p.numel() for p in se_block.parameters()):,}")

### 2.3 SE-ResNet：將 SE 模塊加入 ResNet

In [ ]:
class SEResidualBlock(nn.Module):
    """帶有 SE 模塊的殘差塊"""
    
    def __init__(self, in_channels, out_channels, stride=1, reduction=16):
        super(SEResidualBlock, self).__init__()
        
        # 殘差分支
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # SE 模塊
        self.se = SEBlock(out_channels, reduction)
        
        # 快捷連接
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
    
    def forward(self, x):
        # 主分支
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        
        # SE 模塊
        out = self.se(out)
        
        # 殘差連接
        out += self.shortcut(x)
        out = F.relu(out)
        
        return out

# 測試
block = SEResidualBlock(64, 128, stride=2)
x = torch.randn(2, 64, 32, 32)
output = block(x)
print(f"SE-ResNet Block: {x.shape} → {output.shape}")

### 2.4 SE模塊的效果分析

In [ ]:
def visualize_se_weights(se_block, x):
    """可視化 SE 模塊學到的通道權重"""
    se_block.eval()
    with torch.no_grad():
        b, c, _, _ = x.size()
        
        # 獲取 SE 權重
        y = se_block.squeeze(x).view(b, c)
        weights = se_block.excitation(y).view(b, c).cpu().numpy()
        
        # 繪製權重分布
        plt.figure(figsize=(12, 4))
        
        plt.subplot(1, 2, 1)
        plt.bar(range(c), weights[0])
        plt.xlabel('Channel Index')
        plt.ylabel('SE Weight')
        plt.title('Channel Importance (SE Weights)')
        plt.grid(True, alpha=0.3)
        
        plt.subplot(1, 2, 2)
        plt.hist(weights[0], bins=30, edgecolor='black')
        plt.xlabel('SE Weight')
        plt.ylabel('Frequency')
        plt.title('Distribution of SE Weights')
        plt.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        print(f"權重統計：")
        print(f"  均值: {weights[0].mean():.4f}")
        print(f"  標準差: {weights[0].std():.4f}")
        print(f"  最大值: {weights[0].max():.4f}")
        print(f"  最小值: {weights[0].min():.4f}")

# 可視化
se_block = SEBlock(64, reduction=16)
x = torch.randn(1, 64, 32, 32)
visualize_se_weights(se_block, x)

## 3. MobileNet 系列：輕量級網路

MobileNet 系列專為**移動端和嵌入式設備**設計，核心目標是在保持準確率的同時大幅減少計算量。

### 3.1 MobileNetV1：深度可分離卷積

**論文**: [MobileNets: Efficient Convolutional Neural Networks for Mobile Vision Applications](https://arxiv.org/abs/1704.04861) (2017)

#### 核心創新：Depthwise Separable Convolution

將標準卷積分解為兩步：
1. **Depthwise Convolution**：每個通道獨立卷積
2. **Pointwise Convolution**：1×1 卷積混合通道

**計算量對比**：
- 標準卷積：$H \times W \times C_{in} \times C_{out} \times K^2$
- 深度可分離：$H \times W \times C_{in} \times (K^2 + C_{out})$
- **減少倍數**：約 $\frac{1}{C_{out}} + \frac{1}{K^2}$（通常8-9倍）

In [ ]:
class DepthwiseSeparableConv(nn.Module):
    """深度可分離卷積"""
    
    def __init__(self, in_channels, out_channels, stride=1):
        super(DepthwiseSeparableConv, self).__init__()
        
        # Depthwise: 每個通道獨立卷積
        self.depthwise = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, kernel_size=3, 
                     stride=stride, padding=1, groups=in_channels, bias=False),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True)
        )
        
        # Pointwise: 1×1 卷積混合通道
        self.pointwise = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        x = self.depthwise(x)
        x = self.pointwise(x)
        return x

# 計算量對比
def compare_computation(in_c, out_c, img_size, kernel_size=3):
    """比較標準卷積和深度可分離卷積的計算量"""
    
    # 標準卷積
    standard_ops = img_size * img_size * in_c * out_c * kernel_size**2
    
    # 深度可分離卷積
    depthwise_ops = img_size * img_size * in_c * kernel_size**2
    pointwise_ops = img_size * img_size * in_c * out_c
    separable_ops = depthwise_ops + pointwise_ops
    
    reduction = standard_ops / separable_ops
    
    print(f"\n計算量對比（{in_c}→{out_c}通道，{img_size}×{img_size}圖像）：")
    print(f"  標準卷積: {standard_ops:,} ops")
    print(f"  深度可分離卷積: {separable_ops:,} ops")
    print(f"  減少倍數: {reduction:.2f}x")
    print(f"  節省: {(1 - 1/reduction)*100:.1f}%")

# 示例
compare_computation(in_c=128, out_c=256, img_size=56, kernel_size=3)

# 測試
ds_conv = DepthwiseSeparableConv(64, 128)
x = torch.randn(2, 64, 56, 56)
output = ds_conv(x)
print(f"\n輸入: {x.shape} → 輸出: {output.shape}")

### 3.2 MobileNetV2：反向殘差和線性瓶頸

**論文**: [MobileNetV2: Inverted Residuals and Linear Bottlenecks](https://arxiv.org/abs/1801.04381) (2018)

#### 核心創新：
1. **Inverted Residual**：先擴展再壓縮（與ResNet相反）
2. **Linear Bottleneck**：最後不使用ReLU（保留信息）

```
ResNet瓶頸:     寬 → 窄 → 寬
MobileNetV2:    窄 → 寬 → 窄  (Inverted)
```

In [ ]:
class InvertedResidual(nn.Module):
    """MobileNetV2 的反向殘差塊"""
    
    def __init__(self, in_channels, out_channels, stride, expand_ratio=6):
        """
        Args:
            in_channels: 輸入通道數
            out_channels: 輸出通道數
            stride: 步長
            expand_ratio: 擴展比例（通常是6）
        """
        super(InvertedResidual, self).__init__()
        self.stride = stride
        
        hidden_dim = int(in_channels * expand_ratio)
        self.use_res_connect = (stride == 1 and in_channels == out_channels)
        
        layers = []
        
        # 1. Expansion（擴展）
        if expand_ratio != 1:
            layers.extend([
                nn.Conv2d(in_channels, hidden_dim, 1, bias=False),
                nn.BatchNorm2d(hidden_dim),
                nn.ReLU6(inplace=True)  # ReLU6 = min(max(0, x), 6)
            ])
        
        # 2. Depthwise（深度卷積）
        layers.extend([
            nn.Conv2d(hidden_dim, hidden_dim, 3, stride, 1, 
                     groups=hidden_dim, bias=False),
            nn.BatchNorm2d(hidden_dim),
            nn.ReLU6(inplace=True)
        ])
        
        # 3. Projection（投影，線性瓶頸）
        layers.extend([
            nn.Conv2d(hidden_dim, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels)
            # 注意：這裡沒有 ReLU！
        ])
        
        self.conv = nn.Sequential(*layers)
    
    def forward(self, x):
        if self.use_res_connect:
            return x + self.conv(x)
        else:
            return self.conv(x)

# 對比不同塊結構
def visualize_block_structure():
    """可視化不同塊的結構"""
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # ResNet瓶頸
    axes[0].barh(['Input', 'Conv1×1', 'Conv3×3', 'Conv1×1', 'Output'], 
                 [256, 64, 64, 256, 256], color='skyblue')
    axes[0].set_title('ResNet Bottleneck\n(寬→窄→寬)', fontweight='bold')
    axes[0].set_xlabel('Channels')
    
    # MobileNetV1
    axes[1].barh(['Input', 'DWConv', 'PWConv', 'Output'], 
                 [256, 256, 256, 256], color='lightgreen')
    axes[1].set_title('MobileNetV1\n(深度可分離)', fontweight='bold')
    axes[1].set_xlabel('Channels')
    
    # MobileNetV2
    axes[2].barh(['Input', 'Expand', 'DWConv', 'Project', 'Output'], 
                 [64, 384, 384, 64, 64], color='salmon')
    axes[2].set_title('MobileNetV2\n(窄→寬→窄)', fontweight='bold')
    axes[2].set_xlabel('Channels')
    
    plt.tight_layout()
    plt.show()

visualize_block_structure()

# 測試
block = InvertedResidual(64, 64, stride=1, expand_ratio=6)
x = torch.randn(2, 64, 56, 56)
output = block(x)
print(f"\nInverted Residual: {x.shape} → {output.shape}")

### 3.3 MobileNetV3：神經架構搜索 + 新激活函數

**論文**: [Searching for MobileNetV3](https://arxiv.org/abs/1905.02244) (2019)

#### 核心改進：
1. 使用 **NAS**（神經架構搜索）找最優架構
2. 引入 **h-swish** 激活函數（比ReLU6更好）
3. 加入 **SE模塊**
4. 重新設計頭部和尾部

In [ ]:
class HSwish(nn.Module):
    """Hard Swish 激活函數"""
    def forward(self, x):
        # h-swish(x) = x * ReLU6(x + 3) / 6
        return x * F.relu6(x + 3) / 6

class HSigmoid(nn.Module):
    """Hard Sigmoid 激活函數"""
    def forward(self, x):
        # h-sigmoid(x) = ReLU6(x + 3) / 6
        return F.relu6(x + 3) / 6

# 可視化不同激活函數
def compare_activations():
    x = torch.linspace(-5, 5, 1000)
    
    relu = F.relu(x)
    relu6 = F.relu6(x)
    swish = x * torch.sigmoid(x)
    hswish = x * F.relu6(x + 3) / 6
    
    plt.figure(figsize=(12, 6))
    
    plt.subplot(1, 2, 1)
    plt.plot(x, relu, label='ReLU', linewidth=2)
    plt.plot(x, relu6, label='ReLU6', linewidth=2)
    plt.plot(x, swish, label='Swish', linewidth=2)
    plt.plot(x, hswish, label='h-swish', linewidth=2, linestyle='--')
    plt.xlabel('x')
    plt.ylabel('Activation(x)')
    plt.title('激活函數對比', fontweight='bold')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # 導數
    plt.subplot(1, 2, 2)
    x.requires_grad = True
    for name, func in [('ReLU', F.relu), ('h-swish', lambda x: x * F.relu6(x + 3) / 6)]:
        y = func(x)
        grad = torch.autograd.grad(y.sum(), x, create_graph=True)[0]
        plt.plot(x.detach(), grad.detach(), label=f"{name} gradient", linewidth=2)
    
    plt.xlabel('x')
    plt.ylabel('Gradient')
    plt.title('激活函數梯度對比', fontweight='bold')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

compare_activations()

## 4. EfficientNet：複合縮放的藝術

**論文**: [EfficientNet: Rethinking Model Scaling for Convolutional Neural Networks](https://arxiv.org/abs/1905.11946) (ICML 2019)

### 4.1 核心思想

傳統縮放方法只調整一個維度：
- **深度縮放**：增加層數（如 ResNet-50 → ResNet-101）
- **寬度縮放**：增加通道數
- **分辨率縮放**：增加輸入尺寸

EfficientNet 提出**複合縮放**（Compound Scaling）：
```
depth:      d = α^φ
width:      w = β^φ  
resolution: r = γ^φ

約束: α · β² · γ² ≈ 2
```

### 4.2 EfficientNet 架構

In [ ]:
# 使用 timm 加載 EfficientNet
def explore_efficientnet_family():
    """探索 EfficientNet 系列"""
    
    models = [
        'efficientnet_b0',
        'efficientnet_b1', 
        'efficientnet_b3',
        'efficientnet_b7'
    ]
    
    results = []
    
    print("EfficientNet 系列對比：\n")
    print(f"{'Model':<20} {'Params':<15} {'Input Size':<12} {'Top-1 Acc*'}")
    print("-" * 70)
    
    accuracies = [77.1, 79.1, 81.6, 84.3]  # ImageNet Top-1 準確率
    
    for i, model_name in enumerate(models):
        model = timm.create_model(model_name, pretrained=False)
        params = sum(p.numel() for p in model.parameters()) / 1e6
        
        # 獲取默認配置
        cfg = model.default_cfg
        input_size = cfg.get('input_size', (3, 224, 224))[1]
        
        print(f"{model_name:<20} {params:>6.1f}M {input_size:>8}×{input_size:<8} {accuracies[i]:>6.1f}%")
        
        results.append({
            'model': model_name,
            'params': params,
            'input_size': input_size,
            'accuracy': accuracies[i]
        })
    
    print("\n* ImageNet Top-1 準確率（論文報告）")
    
    return results

# 可視化 EfficientNet 縮放
def visualize_compound_scaling(results):
    """可視化複合縮放效果"""
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    models = [r['model'].replace('efficientnet_', '') for r in results]
    params = [r['params'] for r in results]
    accuracy = [r['accuracy'] for r in results]
    
    # 參數量 vs 準確率
    axes[0].scatter(params, accuracy, s=200, alpha=0.6, c=range(len(models)), cmap='viridis')
    for i, (p, a, m) in enumerate(zip(params, accuracy, models)):
        axes[0].annotate(m.upper(), (p, a), fontsize=12, ha='center')
    axes[0].set_xlabel('參數量 (M)', fontsize=12)
    axes[0].set_ylabel('ImageNet Top-1 準確率 (%)', fontsize=12)
    axes[0].set_title('EfficientNet: 參數效率', fontweight='bold', fontsize=14)
    axes[0].grid(True, alpha=0.3)
    
    # 縮放倍數
    base_params = params[0]
    scaling_factors = [p / base_params for p in params]
    
    x = np.arange(len(models))
    axes[1].bar(x, scaling_factors, color='steelblue', alpha=0.7)
    axes[1].set_xticks(x)
    axes[1].set_xticklabels([m.upper() for m in models])
    axes[1].set_ylabel('相對參數量（以B0為基準）', fontsize=12)
    axes[1].set_title('模型縮放倍數', fontweight='bold', fontsize=14)
    axes[1].grid(True, alpha=0.3, axis='y')
    
    # 添加數值標籤
    for i, v in enumerate(scaling_factors):
        axes[1].text(i, v + 0.3, f'{v:.1f}x', ha='center', fontsize=11)
    
    plt.tight_layout()
    plt.show()

results = explore_efficientnet_family()
visualize_compound_scaling(results)

### 4.3 MBConv 塊（EfficientNet的基本構建塊）

In [ ]:
class MBConvBlock(nn.Module):
    """Mobile Inverted Bottleneck Convolution Block（EfficientNet使用）"""
    
    def __init__(self, in_channels, out_channels, kernel_size, 
                 stride, expand_ratio, se_ratio=0.25):
        super(MBConvBlock, self).__init__()
        
        self.use_residual = (stride == 1 and in_channels == out_channels)
        hidden_dim = in_channels * expand_ratio
        
        # 1. Expansion
        if expand_ratio != 1:
            self.expand_conv = nn.Sequential(
                nn.Conv2d(in_channels, hidden_dim, 1, bias=False),
                nn.BatchNorm2d(hidden_dim),
                nn.SiLU()  # Swish activation
            )
        else:
            self.expand_conv = nn.Identity()
        
        # 2. Depthwise
        self.depthwise_conv = nn.Sequential(
            nn.Conv2d(hidden_dim, hidden_dim, kernel_size, stride, 
                     padding=kernel_size//2, groups=hidden_dim, bias=False),
            nn.BatchNorm2d(hidden_dim),
            nn.SiLU()
        )
        
        # 3. SE (Squeeze-and-Excitation)
        se_channels = max(1, int(in_channels * se_ratio))
        self.se = SEBlock(hidden_dim, reduction=hidden_dim // se_channels)
        
        # 4. Projection
        self.project_conv = nn.Sequential(
            nn.Conv2d(hidden_dim, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels)
        )
    
    def forward(self, x):
        identity = x
        
        x = self.expand_conv(x)
        x = self.depthwise_conv(x)
        x = self.se(x)
        x = self.project_conv(x)
        
        if self.use_residual:
            x = x + identity
        
        return x

# 測試
mbconv = MBConvBlock(in_channels=32, out_channels=32, 
                    kernel_size=3, stride=1, expand_ratio=6)
x = torch.randn(2, 32, 56, 56)
output = mbconv(x)
print(f"MBConv Block: {x.shape} → {output.shape}")
print(f"參數量: {sum(p.numel() for p in mbconv.parameters()):,}")

## 5. 現代架構性能大比拼

In [ ]:
def comprehensive_model_comparison():
    """全面的模型對比"""
    
    models_info = [
        # (model_name, top1_acc, year, category)
        ('resnet50', 76.1, 2015, 'Classic'),
        ('se_resnet50', 77.6, 2018, 'Attention'),
        ('resnext50_32x4d', 77.6, 2017, 'Architecture'),
        ('mobilenetv2_100', 72.0, 2018, 'Mobile'),
        ('mobilenetv3_large_100', 75.2, 2019, 'Mobile'),
        ('efficientnet_b0', 77.1, 2019, 'NAS'),
        ('efficientnet_b3', 81.6, 2019, 'NAS'),
    ]
    
    results = []
    
    print("現代CNN架構全面對比\n")
    print(f"{'Model':<25} {'Params':>10} {'Top-1':>8} {'Year':>6} {'Category':>12}")
    print("-" * 75)
    
    for model_name, top1, year, category in models_info:
        try:
            model = timm.create_model(model_name, pretrained=False)
            params = sum(p.numel() for p in model.parameters()) / 1e6
            
            print(f"{model_name:<25} {params:>8.1f}M {top1:>7.1f}% {year:>6} {category:>12}")
            
            results.append({
                'name': model_name,
                'params': params,
                'top1': top1,
                'year': year,
                'category': category
            })
        except Exception as e:
            print(f"Error loading {model_name}: {e}")
    
    return results

def visualize_model_comparison(results):
    """可視化模型對比"""
    
    fig = plt.figure(figsize=(16, 10))
    gs = fig.add_gridspec(2, 2, hspace=0.3, wspace=0.3)
    
    # 1. 參數量 vs 準確率
    ax1 = fig.add_subplot(gs[0, 0])
    categories = list(set(r['category'] for r in results))
    colors = plt.cm.Set3(np.linspace(0, 1, len(categories)))
    category_colors = {cat: colors[i] for i, cat in enumerate(categories)}
    
    for result in results:
        ax1.scatter(result['params'], result['top1'], 
                   s=200, alpha=0.6, 
                   c=[category_colors[result['category']]],
                   label=result['category'])
        ax1.annotate(result['name'].replace('_', '\n'), 
                    (result['params'], result['top1']),
                    fontsize=8, ha='center', alpha=0.8)
    
    # 去重圖例
    handles, labels = ax1.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    ax1.legend(by_label.values(), by_label.keys(), loc='lower right')
    
    ax1.set_xlabel('Parameters (M)', fontsize=12)
    ax1.set_ylabel('ImageNet Top-1 Accuracy (%)', fontsize=12)
    ax1.set_title('模型效率：參數量 vs 準確率', fontweight='bold', fontsize=14)
    ax1.grid(True, alpha=0.3)
    
    # 2. 每類別最佳模型
    ax2 = fig.add_subplot(gs[0, 1])
    category_best = {}
    for result in results:
        cat = result['category']
        if cat not in category_best or result['top1'] > category_best[cat]['top1']:
            category_best[cat] = result
    
    cats = list(category_best.keys())
    accs = [category_best[cat]['top1'] for cat in cats]
    
    bars = ax2.barh(cats, accs, color=[category_colors[cat] for cat in cats], alpha=0.7)
    ax2.set_xlabel('Top-1 Accuracy (%)', fontsize=12)
    ax2.set_title('各類別最佳模型', fontweight='bold', fontsize=14)
    ax2.grid(True, alpha=0.3, axis='x')
    
    # 添加數值標籤
    for i, (cat, acc) in enumerate(zip(cats, accs)):
        model_name = category_best[cat]['name']
        ax2.text(acc + 0.5, i, f"{model_name}\n{acc:.1f}%", 
                va='center', fontsize=9)
    
    # 3. 時間演進
    ax3 = fig.add_subplot(gs[1, :])
    
    for cat in categories:
        cat_results = [r for r in results if r['category'] == cat]
        cat_results.sort(key=lambda x: x['year'])
        
        years = [r['year'] for r in cat_results]
        accs = [r['top1'] for r in cat_results]
        
        ax3.plot(years, accs, marker='o', linewidth=2, 
                label=cat, color=category_colors[cat], markersize=8)
    
    ax3.set_xlabel('Year', fontsize=12)
    ax3.set_ylabel('ImageNet Top-1 Accuracy (%)', fontsize=12)
    ax3.set_title('CNN 架構演進（2015-2019）', fontweight='bold', fontsize=14)
    ax3.legend(loc='lower right')
    ax3.grid(True, alpha=0.3)
    
    plt.show()

# 執行對比
results = comprehensive_model_comparison()
print("\n生成可視化對比圖...")
visualize_model_comparison(results)

## 6. 實戰：選擇合適的架構

### 6.1 決策樹

In [ ]:
def recommend_architecture(use_case):
    """
    根據使用場景推薦架構
    
    use_case: 'mobile', 'cloud', 'edge', 'research'
    """
    
    recommendations = {
        'mobile': {
            'primary': 'mobilenetv3_small_100',
            'alternative': ['mobilenetv2_100', 'efficientnet_b0'],
            'reason': '輕量級，專為移動端優化',
            'tips': [
                '使用量化以減少模型大小',
                '考慮使用 TensorFlow Lite 或 PyTorch Mobile',
                '測試實際設備上的推理速度'
            ]
        },
        'edge': {
            'primary': 'efficientnet_b0',
            'alternative': ['mobilenetv3_large_100', 'resnet18'],
            'reason': '平衡準確率和效率',
            'tips': [
                '使用混合精度（FP16）',
                '考慮模型剪枝',
                '批處理以提高吞吐量'
            ]
        },
        'cloud': {
            'primary': 'efficientnet_b7',
            'alternative': ['resnet152', 'se_resnext101_32x4d'],
            'reason': '追求最高準確率',
            'tips': [
                '使用分布式訓練',
                '數據並行加速',
                '充分的數據增強'
            ]
        },
        'research': {
            'primary': 'resnet50',
            'alternative': ['efficientnet_b3', 'mobilenetv3_large_100'],
            'reason': '經典架構，便於對比和改進',
            'tips': [
                '從預訓練模型開始',
                '記錄詳細的實驗日誌',
                '嘗試不同的修改和改進'
            ]
        }
    }
    
    if use_case not in recommendations:
        print(f"未知場景: {use_case}")
        print(f"可選: {list(recommendations.keys())}")
        return
    
    rec = recommendations[use_case]
    
    print(f"\n📱 使用場景: {use_case.upper()}")
    print(f"\n✅ 推薦模型: {rec['primary']}")
    print(f"   理由: {rec['reason']}")
    print(f"\n🔄 備選方案: {', '.join(rec['alternative'])}")
    print(f"\n💡 實用建議:")
    for i, tip in enumerate(rec['tips'], 1):
        print(f"   {i}. {tip}")
    
    # 加載並顯示模型信息
    try:
        model = timm.create_model(rec['primary'], pretrained=False)
        params = sum(p.numel() for p in model.parameters()) / 1e6
        print(f"\n📊 模型統計:")
        print(f"   參數量: {params:.2f}M")
    except:
        pass

# 示例
print("=" * 60)
recommend_architecture('mobile')
print("\n" + "=" * 60)
recommend_architecture('cloud')

## 7. 小結與展望

### 7.1 關鍵技術總結

| 技術 | 代表架構 | 核心思想 | 優勢 |
|------|---------|----------|------|
| **通道注意力** | SENet | 學習通道重要性 | 小開銷大提升 |
| **深度可分離卷積** | MobileNet | 分解卷積操作 | 大幅減少計算量 |
| **複合縮放** | EfficientNet | 平衡深度/寬度/分辨率 | 最優參數效率 |
| **神經架構搜索** | EfficientNet/MobileNetV3 | 自動設計架構 | 超越人工設計 |
| **反向殘差** | MobileNetV2 | 先擴展後壓縮 | 保留更多信息 |

### 7.2 選擇指南

```python
if 追求最高準確率:
    選擇: EfficientNet-B7, SE-ResNeXt-101
    
elif 移動端部署:
    選擇: MobileNetV3, EfficientNet-B0
    
elif 平衡性能:
    選擇: ResNet-50, EfficientNet-B3
    
elif 研究和教學:
    選擇: ResNet-50（經典，易於理解）
```

### 7.3 未來趨勢

1. **Vision Transformer 崛起**
   - ViT, Swin Transformer 超越 CNN
   - 但 CNN 仍在移動端和邊緣計算占優

2. **混合架構**
   - CNN + Transformer（如 CoAtNet）
   - 結合兩者優勢

3. **持續優化**
   - 更高效的架構搜索
   - 量化感知訓練
   - 模型壓縮技術

### 7.4 實踐建議

1. **從預訓練開始**
   - 幾乎總是比從頭訓練好
   - 使用 timm 庫快速實驗

2. **根據資源選擇**
   - GPU充足 → 大模型
   - 資源受限 → 輕量模型

3. **性能評估**
   - 不只看準確率
   - 測量實際推理速度
   - 考慮內存佔用

4. **持續學習**
   - 關注最新論文
   - 參與開源社區
   - 實踐項目應用

## 8. 練習題

1. **理論題**
   - 解釋為什麼深度可分離卷積能大幅減少計算量
   - 比較 ResNet 瓶頸塊和 MobileNetV2 反向殘差塊的異同
   - EfficientNet 的複合縮放為什麼比單一縮放更有效？

2. **實作題**
   - 實現一個完整的 SE-ResNet-18
   - 在 CIFAR-10 上對比 MobileNetV2 和 ResNet-18 的性能
   - 使用 EfficientNet-B0 進行遷移學習

3. **進階題**
   - 設計一個混合架構（結合 SE 和 MBConv）
   - 實現模型量化並對比推理速度
   - 在實際項目中應用並部署一個現代CNN模型

## 參考資源

### 論文
- [SENet](https://arxiv.org/abs/1709.01507)
- [MobileNetV1](https://arxiv.org/abs/1704.04861)
- [MobileNetV2](https://arxiv.org/abs/1801.04381)
- [MobileNetV3](https://arxiv.org/abs/1905.02244)
- [EfficientNet](https://arxiv.org/abs/1905.11946)

### 代碼庫
- [timm (PyTorch Image Models)](https://github.com/rwightman/pytorch-image-models)
- [TensorFlow Models](https://github.com/tensorflow/models)

### 工具
- [Netron](https://netron.app/) - 模型可視化
- [Papers with Code](https://paperswithcode.com/) - 論文與代碼